# Notebook 2: Data Cleaning
**Project:** Car Price Estimation  
**Goal:** Handle missing values, duplicates, inconsistent values, data types, outliers, and standardize text.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('../data/CarPrice_Assignment.csv')
print(f'Loaded: {df.shape}')

## 1. Handle Missing Values

In [ ]:
print('Missing values per column:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')

# Fill missing numerics with median, categoricals with mode
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['float64', 'int64']:
            df[col].fillna(df[col].median(), inplace=True)
            print(f'Filled {col} with median')
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)
            print(f'Filled {col} with mode')

print(f'\nMissing after fix: {df.isnull().sum().sum()}')

## 2. Check and Handle Duplicates

In [ ]:
dups = df.duplicated().sum()
print(f'Duplicate rows: {dups}')
if dups > 0:
    df.drop_duplicates(inplace=True)
    print(f'Duplicates removed. New shape: {df.shape}')
else:
    print('No duplicates found.')

## 3. Fix Inconsistent CarName / Brand Values

In [ ]:
# Standardize: strip whitespace, lowercase
df['CarName'] = df['CarName'].str.strip().str.lower()

# Extract brand from CarName (first word)
df['brand'] = df['CarName'].apply(lambda x: x.split()[0])

print('Unique brands before fix:')
print(sorted(df['brand'].unique()))

In [ ]:
# Fix known typos and inconsistencies
brand_corrections = {
    'maxda'    : 'mazda',
    'toyouta'  : 'toyota',
    'vokswagen': 'volkswagen',
    'vw'       : 'volkswagen',
    'porcshce' : 'porsche',
    'alfa-romero': 'alfa-romeo'
}
df['brand'] = df['brand'].replace(brand_corrections)

print('Unique brands after fix:')
print(sorted(df['brand'].unique()))
print(f'\nTotal unique brands: {df["brand"].nunique()}')

## 4. Correct Data Types

In [ ]:
print('Data types before:')
print(df.dtypes)

# Ensure numeric columns are numeric
numeric_cols = ['wheelbase','carlength','carwidth','carheight','curbweight',
                'enginesize','boreratio','stroke','compressionratio',
                'horsepower','peakrpm','citympg','highwaympg','price']

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print('\nData types after:')
print(df.dtypes)

## 5. Handle Outliers in Numerical Columns

In [ ]:
def detect_outliers_iqr(df, col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    return outliers, lower, upper

print('Outlier Summary:')
print(f'{"Column":<20} {"Outliers":<10} {"Lower":<12} {"Upper":<12}')
print('-'*55)
for col in numeric_cols:
    if col in df.columns:
        out, low, up = detect_outliers_iqr(df, col)
        print(f'{col:<20} {len(out):<10} {low:<12.2f} {up:<12.2f}')

In [ ]:
# Cap outliers using IQR (Winsorization) - preserve data, cap extremes
cols_to_cap = ['price', 'horsepower', 'enginesize', 'curbweight']
for col in cols_to_cap:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    before = df[col].describe()
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f'Capped {col}: [{lower:.1f}, {upper:.1f}]')

print('\nOutlier capping complete.')

In [ ]:
# Visualize boxplots after cleaning
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
plot_cols = ['price', 'horsepower', 'enginesize', 'curbweight', 'carlength', 'carwidth']
for ax, col in zip(axes.flatten(), plot_cols):
    ax.boxplot(df[col].dropna(), patch_artist=True, boxprops=dict(facecolor='steelblue', alpha=0.7))
    ax.set_title(f'{col}', fontsize=12)
    ax.set_ylabel('Value')
plt.suptitle('Boxplots After Outlier Treatment', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../reports/02_boxplots_after_cleaning.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Standardize All Text Columns

In [ ]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    df[col] = df[col].str.strip().str.lower()
    print(f'{col}: {df[col].unique()}')

## 7. Final Check & Save Cleaned Dataset

In [ ]:
print('='*60)
print('CLEANED DATASET SUMMARY')
print('='*60)
print(f'Shape          : {df.shape}')
print(f'Missing values : {df.isnull().sum().sum()}')
print(f'Duplicates     : {df.duplicated().sum()}')
print(f'Brands         : {sorted(df["brand"].unique())}')
df.head()

In [ ]:
df.to_csv('../data/02_cleaned_data.csv', index=False)
print('Cleaned dataset saved to data/02_cleaned_data.csv')
print('Notebook 2 Complete!')